In [1]:
import numpy as np
from scipy.linalg import expm, pinvh, LinAlgError
from scipy.special import comb
from scipy.integrate import *

import matplotlib.pyplot as plt

from electron_integrals import *
from CI import *

Define number of electrons and orbitals

In [2]:
# Number of orbitals (without spin)
num_orbitals = 3
# Number of electrons
num_electrons = 1
#Include spin?
include_spin = False

spin_factor = 1+int(include_spin)

num_spin_orbitals = spin_factor*num_orbitals

Calculate electron integrals

In [3]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

#pot = GaussianWell(w=100, a=1, center=0)
pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a = 0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

#Chemistry convention... (Following Hochstuhl)
g = g.transpose(0,2,1,3)

#g = g - g.transpose(0, 1, 3, 2) anti-symmetrisation, code further down not written for this

print('Sanity test, due to symmetry in g this should be zero:')
print(-g[2,1,1,0]+g[2,1,0,1]+g[1,2,1,0]-g[1,2,0,1])

Sanity test, due to symmetry in g this should be zero:
0.0


Solve using Slater-Condon 

In [4]:
z = np.zeros_like(g)

H = SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E, C = np.linalg.eigh(H)
print(E)
print(C[:,0].T@H@C[:,0])

#D, d = get_RDMs(num_spin_orbitals, num_electrons, C[:,0])
#print(np.einsum('pq, pq', D, h) + 0.5 * np.einsum('pqrs, pqrs', z, d))


[0.49998747 1.49993737 2.49983716]
(0.49998747464902393+0j)


# Orbital equations

Define number of MCTDHF orbitals $(\{\ket{\phi_n(t)}\})$, initial b's (basis reduction coefficients),

$\ket{\phi_n(t)} = \sum_{k=1}^{N_b}b_{nk}(t)\ket{\psi_k}$,

and initial C's.


In [5]:
num_mctdhf_orbitals = 2
num_mctdhf_slater_dets = int(comb(num_mctdhf_orbitals, num_electrons))

# Distributed vales
#b_init = np.zeros((num_mctdhf_orbitals, num_spin_orbitals))
#a = int(num_spin_orbitals/num_mctdhf_orbitals)
#for i in range(num_mctdhf_orbitals):
#    b_init[i,a*i:a*(i+1)] = np.ones(a)

# Identity
#b_init=np.eye(num_mctdhf_orbitals, num_spin_orbitals)

# Generate random orthonormal vectors
rng = np.random.default_rng()
r = rng.random((num_mctdhf_orbitals, num_spin_orbitals))
u, _, vh = np.linalg.svd(r, full_matrices=False)
b_init = (u@vh).astype(np.cdouble)

print(b_init)

# Ones as initial C
#C_init = np.ones((num_mctdhf_slater_dets), dtype=np.cdouble)

# Random initial C
C_init = rng.random(num_mctdhf_slater_dets).astype(np.cdouble)

#Normalize
C_init = np.divide(C_init, np.sqrt(C_init.T.conj()@C_init))
print(C_init)

[[ 0.10466023+0.j  0.8812974 +0.j  0.46082657+0.j]
 [ 0.92835404+0.j -0.25275454+0.j  0.2725324 +0.j]]
[0.99834404+0.j 0.05752544+0.j]


Define functions

In [6]:
# Function to get reduced density matrices

def get_RDMs(num_orbitals, num_electrons, C):
    slater_dets = CIHamiltonian.get_slater_dets(num_orbitals, num_electrons)
    C_conj = C.conj()

    D = np.zeros((num_orbitals,)*2 , dtype=np.cdouble)
    d = np.zeros((num_orbitals,)*4, dtype=np.cdouble)

    for n, det_n in enumerate(slater_dets):
        for m, det_m in enumerate(slater_dets):
            num_differences = np.sum(np.abs(det_n-det_m))
            match(num_differences):
                case 0:
                    for p, n_p in enumerate(det_n):
                        D[p,p] += n_p*C_conj[m]*C[n]
                        for r, n_r in enumerate(det_n):
                            d[p,p,r,r] += n_p*n_r*C_conj[m]*C[n]
                            d[p,r,r,p] -= n_p*n_r*C_conj[m]*C[n] # Note the sign                 
                case 2:
                    p = np.flatnonzero(np.asarray((det_m-det_n)==1))[0]
                    q = np.flatnonzero(np.asarray((det_n-det_m)==1))[0]
                    gamma = (-1)**(np.sum(det_n[:q])+np.sum(det_m[:p]))
                    D[p,q] += gamma*C_conj[m]*C[n]

                    for r, n_r in enumerate(det_n):
                        val = gamma*n_r*C_conj[m]*C[n]
                        d[p,q,r,r] += val
                        d[r,q,p,r] -= val # Note the sign
                        d[p,r,r,q] -= val # Note the sign
                        d[r,r,p,q] += val

                case 4:
                    p,r = np.flatnonzero(np.asarray((det_m-det_n)==1))
                    q,s = np.flatnonzero(np.asarray((det_n-det_m)==1))
                    
                    gamma = np.sum(det_n[:q])
                    gamma += np.sum(det_n[:s])-1 # -1 since q<s 
                    gamma += np.sum(det_n[:r])-int(s<r)-int(q<r)
                    gamma += np.sum(det_n[:p])-int(s<p)-int(q<p) # No additional term since p<r

                    val = C_conj[m]*C[n]*(-1)**gamma
                    d[p,q,r,s] += val
                    d[r,q,p,s] -= val # Note the sign
                    d[p,s,r,q] -= val # Note the sign
                    d[r,s,p,q] += val

    return D, d


# Helper functions to put the problem on a form handled by standard SciPy ODE solvers 

def Cb_to_y(C, b):
    return np.concatenate([C.flatten(), b.flatten()])

def y_to_Cb(y, num_mctdhf_orbitals, num_spin_orbitals, num_electrons):
    num_slater_dets = int(comb(num_mctdhf_orbitals, num_electrons))

    C, b = np.split(y, [num_slater_dets])
    b = b.reshape((num_mctdhf_orbitals, num_spin_orbitals))

    return C, b

Define callable class for the differential equations

In [7]:
class MCDTHF_imaginary_time:
    def __init__(self, h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons):
        self.h = h
        self.g = g
        self.num_mctdhf_orbitals = num_mctdhf_orbitals
        self.num_spin_orbitals = num_spin_orbitals
        self.num_electrons = num_electrons

    def __call__(self, t, y):
        
        C, b = y_to_Cb(y, self.num_mctdhf_orbitals, self.num_spin_orbitals, self.num_electrons)
        bc = b.conj()

        h_1 = np.einsum('nk, jk -> nj', b, self.h)
        h_2 = np.einsum('mj, nj -> nm', bc, h_1) # Fully transformed integral
        h_3 = np.einsum('mj, nm -> nj', b, h_2)

        g_2 = np.einsum('rk, sl, ijkl -> ijrs', bc, b, self.g)
        g_3 = np.einsum('qj, ijrs -> iqrs', b, g_2)
        g_4 = np.einsum('pi, iqrs -> pqrs', bc, g_3) # Fully transformed integral
        g_5 = np.einsum('pi, pqrs -> iqrs', b, g_4)

        D, d = get_RDMs(num_mctdhf_orbitals, num_electrons, C)

        # Regularize
        #eps = 1e-3
        #D_reg = D + eps*expm(-D/eps)
        
        H = SlaterCondonHamiltonian(self.num_mctdhf_orbitals, self.num_electrons, h_2, g_4).get_hamiltonian()
        
        C_dot = -np.matmul(H,C)
        C_dot = np.round(C_dot, 7)

        try:
            b_dot = -(h_1-h_3 + np.einsum('np, pqrs, kqrs -> nk', np.linalg.pinv(D, hermitian = True), d, g_3-g_5))
            b_dot = np.round(b_dot, 7)
        except (LinAlgError, np.linalg.LinAlgError):
            print('SVD error!')
            print(D)
            raise

        return Cb_to_y(C_dot, b_dot)

Solve using ODE solver

In [8]:
#mctdhf_fun = MCDTHF(h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons)
mctdhf_fun = MCDTHF_imaginary_time(h, z, num_mctdhf_orbitals, num_spin_orbitals, num_electrons)

y_init = Cb_to_y(C_init, b_init.T)

Cs = np.copy(C_init).reshape(1,-1)

t_init = 0
t_final = 20.0
solver = DOP853(mctdhf_fun, t_init, y_init, t_final)
#solver = RK45(mctdhf_fun, t_init, y_init, t_final)

while solver.status == 'running':
    msg = solver.step()

    C, b = y_to_Cb(solver.y, num_mctdhf_orbitals, num_spin_orbitals, num_electrons)
    # Normalize every step during imaginary time prop
    C = np.divide(C, np.sqrt(C.T.conj()@C))
    #b = np.divide(b.T, np.sqrt(np.einsum('ij, ij -> i', b.conj(), b))).T
    
    b = np.divide(b, np.sqrt(np.einsum('ij, ij -> j', b.conj(), b)))
    h_tilde = np.einsum('kp, jq, kj -> pq', b.conj(), b, h)

    C=np.round(C, 7)
    b = np.round(b, 7)

    D, d = get_RDMs(num_mctdhf_orbitals, num_electrons, C)
    #h_tilde = np.einsum('pk, qj, kj -> pq', b.conj(), b, h)
    #g_tilde = np.einsum('pi, qj, rk, sl, ijkl -> pqrs', b.conj(), b.conj(), b, b, g)
    print(f"E = {np.einsum('pq, pq', D, h_tilde)}")# + 0.5 * np.einsum('pqrs, pqrs', g_tilde, d)}")
    Cs = np.append(Cs, C.reshape(1,-1), axis=0)

    solver.y = Cb_to_y(C, b)
else:
    if solver.status != 'finished':
        raise(RuntimeError(msg))

ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (2,3)->(3,newaxis,newaxis,2) (2,3)->(3,2,newaxis) (3,3)->(3,3) 